In [2]:
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

In [3]:
data = joblib.load("../../data/03_gold/fundamentals.pkl")
X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]
usable_features = data["usable_features"]
X_train

ItemName,EntityCode,Date,TotalEquityGrossMinorityInterest,TotalAssets,TotalLiabilitiesNetMinorityInterest,CommonStockEquity,StockholdersEquity,TaxEffectOfUnusualItems,NetTangibleAssets,TaxRateForCalcs,...,LossAdjustmentExpense,DuefromRelatedPartiesNonCurrent,CashFlowFromDiscontinuedOperation,DuetoRelatedPartiesNonCurrent,UnrealizedGainLoss,TotalPartnershipCapital,PolicyholderBenefitsGross,LimitedPartnershipCapital,InterestReceivedDirect,PolicyholderBenefitsCeded
1,0001.HK,2022-12-31,8.304796e+10,1.473413e+11,6.429334e+10,6.775044e+10,6.775044e+10,3.253294e+08,1.484181e+10,0.020440,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0001.HK,2023-12-31,8.583796e+10,1.483529e+11,6.251491e+10,7.022722e+10,7.022722e+10,2.010523e+07,1.661653e+10,0.011575,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0002.HK,2022-12-31,1.484322e+10,3.028148e+10,1.543826e+10,1.403379e+10,1.403379e+10,-3.950256e+07,1.083586e+10,0.008311,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0002.HK,2023-12-31,1.438618e+10,2.932115e+10,1.493497e+10,1.359712e+10,1.359712e+10,-2.146585e+08,1.195166e+10,0.035759,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,0003.HK,2022-12-31,9.640088e+09,2.161415e+10,1.197406e+10,8.161206e+09,8.161206e+09,3.692969e+06,7.165632e+09,0.029147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38710,ZVRA,2023-12-31,6.186400e+07,1.723270e+08,1.104630e+08,6.186400e+07,6.186400e+07,0.000000e+00,-1.206400e+07,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38714,ZWS,2022-12-31,1.615000e+09,2.864000e+09,1.249000e+09,1.615000e+09,1.615000e+09,-4.928000e+06,-1.717000e+08,0.320000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38715,ZWS,2023-12-31,1.602800e+09,2.667000e+09,1.064200e+09,1.602800e+09,1.602800e+09,-8.009264e+06,-1.456000e+08,0.290191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38719,ZYME,2022-12-31,4.929560e+08,6.487250e+08,1.557690e+08,4.929560e+08,4.929560e+08,9.279276e+04,4.545570e+08,0.080549,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
feature_mean = X_train[usable_features].mean()
feature_std  = X_train[usable_features].std()

def scale(df):
    return (df[usable_features] - feature_mean) / feature_std

X_train = scale(X_train).fillna(0.0).values
X_val   = scale(X_val).fillna(0.0).values
X_test  = scale(X_test).fillna(0.0).values
X_train.max()
#M_train = X_train[usable_features].notna().astype(int).values
#M_val   = X_val[usable_features].notna().astype(int).values
#M_test  = X_test[usable_features].notna().astype(int).values

110.43866404997962

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from typing import Optional, Tuple

def prepare_financial_data_with_mask(
    df: pd.DataFrame, 
    usable_features: list, 
    scaler: Optional[RobustScaler] = None
) -> Tuple[np.ndarray, np.ndarray, RobustScaler]:
    """
    Transformiert Finanzdaten via arcsinh und RobustScaler und erzeugt
    eine binäre Maske für Missing Value Embeddings.
    
    Wenn 'scaler' None ist, wird ein neuer RobustScaler gefittet (Train-Modus).
    Wenn ein 'scaler' übergeben wird, wird dieser wiederverwendet (Val/Test-Modus).
    """
    # 1. Asinh-Transformation (bleibt NaN-erhaltend)
    df_transformed = np.arcsinh(df[usable_features])
    
    # 2. Binäre Maske vor der Imputation erstellen (1.0 = Vorhanden, 0.0 = Fehlt)
    mask = (~df_transformed.isna()).astype(np.float32).values
    
    # 3. RobustScaler anwenden
    if scaler is None:
        # Trainings-Modus: Scaler neu anpassen
        scaler = RobustScaler()
        scaled_array = scaler.fit_transform(df_transformed)
    else:
        # Val/Test-Modus: Existierenden Scaler nutzen
        scaled_array = scaler.transform(df_transformed)
    
    # 4. NaNs im skalierten Array sauber durch 0.0 ersetzen (für neutrale Imputation)
    scaled_array = np.nan_to_num(scaled_array, nan=0.0).astype(np.float32)
    
    return scaled_array, mask, scaler
# 1. Train-Set verarbeiten (erstellt den fitted_scaler)
X_train_scaled, X_train_mask, fitted_scaler = prepare_financial_data_with_mask(
    df=X_train, 
    usable_features=usable_features, 
    scaler=None
)

# 2. Validation-Set verarbeiten (übergibt den fitted_scaler)
X_val_scaled, X_val_mask, _ = prepare_financial_data_with_mask(
    df=X_val, 
    usable_features=usable_features, 
    scaler=fitted_scaler
)

# 3. Test-Set verarbeiten (falls vorhanden, gleiches Prinzip)
X_test_scaled, X_test_mask, _ = prepare_financial_data_with_mask(
    df=X_test, 
    usable_features=usable_features, 
    scaler=fitted_scaler
)

In [41]:
X_train_scaled

array([[ 1.6468035 ,  1.5246866 ,  1.3479595 , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.656001  ,  1.5265622 ,  1.3410755 , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.1675274 ,  1.0909352 ,  0.9978523 , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.5479886 ,  0.42488512,  0.34146565, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.21979491,  0.03733066, -0.13012017, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.20342825,  0.00704766, -0.20230734, ...,  0.        ,
         0.        ,  0.        ]], dtype=float32)

In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class VAEOutput:
    z: torch.Tensor
    mu: torch.Tensor
    std: torch.Tensor
    x_recon: torch.Tensor
    loss: torch.Tensor
    loss_recon: torch.Tensor
    loss_kl: torch.Tensor

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=512, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_std = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        # Softplus + epsilon for stable std deviation
        std = F.softplus(self.fc_std(h)) + 1e-6
        return mu, std

    def reparameterize(self, mu, std):
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, kl_weight=1.0):
        mu, std = self.encode(x)
        z = self.reparameterize(mu, std)
        x_recon = self.decode(z)

        # 1. Reconstruction Loss (Binary Cross Entropy for MNIST)
        # Sum over features, mean over batch
        recon_loss = F.binary_cross_entropy_with_logits(x_recon, x, reduction='none').sum(dim=1).mean()

        # 2. KL Divergence
        # Analytic KL for Normal distributions
        kl_loss = -0.5 * torch.sum(1 + torch.log(std**2) - mu**2 - std**2, dim=1).mean()

        # 3. Total Loss (ELBO)
        loss = recon_loss + (kl_weight * kl_loss)

        return VAEOutput(z, mu, std, x_recon, loss, recon_loss, kl_loss)

# --- Training Loop Example ---
def train_step(model, batch, optimizer, kl_weight=1.0):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    output = model(batch, kl_weight)

    # Backward pass
    output.loss.backward()

    # Gradient clipping (recommended)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    return output.loss.item()


In [ ]:
"""
VAE mit gelernten Missing-Value-Embeddings für finanzielle Panel-Daten.

Erwartete Inputs (aus der vorherigen Datenvorbereitung):
- X_train, X_val, X_test: np.ndarray, shape (N, D)
    skalierte Feature-Werte, fehlende Werte mit 0.0 gefüllt
    (Skalierung: mean/std NUR aus X_train berechnet, siehe vorheriger Schritt)
- M_train, M_val, M_test: np.ndarray, shape (N, D)
    Missingness-Maske, 1 = beobachtet, 0 = fehlend

Idee:
- Jedes Feature bekommt eine eigene lineare "Werte-Embedding"-Projektion.
- Für fehlende Werte (mask == 0) wird diese Projektion durch einen gelernten,
  pro Feature geteilten "Missing"-Embedding-Vektor ersetzt (analog zu einem
  Mask-Token in Transformern) statt den künstlichen 0-Platzhalter zu embedden.
- Der Reconstruction-Loss wird nur über tatsächlich beobachtete Zellen berechnet
  (maskierter Loss) - der Decoder wird also nie dafür bestraft, dass er
  ursprünglich fehlende Werte "falsch" rekonstruiert.
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#Datenvorbereitungsklasse, kann auch direkt durch TensorDataset() PyTorch Klasse gemacht werden, aber so kann man später noch Anpassungen machen oder die Transformation noch dort reinpacken. 
class PanelDataset(Dataset):
    def __init__(self, X, M): # Objekt kann nur mit Features und Maske initialisiert werden
        self.X = torch.as_tensor(X, dtype=torch.float32) #Wandelt in Tensoren mit 32Bit Floats um. Tensoren sind im Grunde das gleiche wie Arrays aber sie können auf GPU rechnen und PyTorch zeichnet alle mathematischen Operationen in dynamischen Rechengraph auf Tensoren auf -> Es kann somit besser Gradienten berechnen  
        self.M = torch.as_tensor(M, dtype=torch.float32)

    def __len__(self):
        return len(self.X) #Für Batchsize usw braucht man die Länge des Datensatzes

    def __getitem__(self, idx):
        return self.X[idx], self.M[idx] # Damit man sicher sein kann, dass Beispiel und Maske immer zusammenpassen


class FeatureEmbedding(nn.Module):
    """Pro Feature eine Werte-Embedding-Projektion + ein gelerntes Missing-Embedding."""

    def __init__(self, num_features: int, emb_dim: int):
        super().__init__() #Das ist der Konstruktoraufruf für die nn.Module Elternklasse. Hier werden z.B. die Parameter in geeigneten Datenstrukturen für den Optimuzer gespeichert.
        self.num_features = num_features #Übergabe der Anzahl der Features das braucht man um gemeinsam mit der Embedding Dimension die Anzahl der Parameter zu bestimmen die für die Erstellung des Input Embeddings benötig werden 
        self.emb_dim = emb_dim # Hier legt man die Embedding Dimension fest also in was für einer Dimension soll das Feature dargestellt werden.

        # Pro Feature eigene lineare Projektion: x_d -> R^emb_dim
        self.value_weight = nn.Parameter(torch.randn(num_features, emb_dim) * 0.1) # nn.Parameters baut die Gewichtsmatrizen auf die dann später für die Transformation der Input Embeddings verwendet wird. nn.Parameter ist dafür nötig weil der Tensor sonst nicht als Gewichtsmatrix erkannt wird und vom Modell einfach ignoriert würde. 0.1 dient dazu das die Werte möglichst nahe bei 0 initialisiert werden für stabiles Training
        self.value_bias = nn.Parameter(torch.zeros(num_features, emb_dim)) # Genau das gleiche wie value_weight nur für die Biasmatrix. Hier wird jedoch direkt alles mit 0 initialisiert, da der Bias ja addiert wird.

        # Ein gelernter Embedding-Vektor pro Feature für "fehlt"
        self.missing_embedding = nn.Parameter(torch.randn(num_features, emb_dim) * 0.1) # Hier dann nochmal für die Missing Value Embeddings. Der Shape ist immer num_features x emb_dim, damit jedes Feature einen Vektor der Embedding Dimension hat.

    def forward(self, x, mask):
        # x, mask: (batch, num_features)
        value_emb = x.unsqueeze(-1) * self.value_weight + self.value_bias  # (batch, D, emb_dim)  Das unsqueeze fügt hinten eine Dimension an. Dann wird durch das Broadcasting Verfahren die Tensoren von recht nach links angelichen sodass erkannt wird, dass es keine Batch Dimension gibt und diese deshalb passend zur Batchsize kopiert wird, damit jedes Sample die Gewichtsmatrix hat. Dann wird die Skalar Inputs mit der Embedding Transformation multipliziert und somit kommt dann die emb_dim dabei heraus.  
        mask_exp = mask.unsqueeze(-1)  # (batch, D, 1) #Gleiches beim Mask Tensor es wird eine Dimension hinten angehängt
        combined = mask_exp * value_emb + (1 - mask_exp) * self.missing_embedding # Hier werden dann als erste ein Tensor erstellt der alle tatsächlich enthaltetenen Werte transformiert und die nicht Missing auf 0 setzt und dann im zweiten Schritte werden die Missing Embeddings an die entsprechenden Stellen aufaddiert.
        return combined.flatten(start_dim=1)  # (batch, D * emb_dim) Zurückgegeben wird dann wieder ein 2D Tensor, da die Embeddings nun einfach nacheinander aufgereiht werden, damit sie als einzelne Inputs für eine Lineare Schicht genutzt werden können.


class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__() #Wieder der Konstruktoraufruf für das nn.Module zur Vererbung und initialisierung wichtiger Parts
        layers = [] # Speicher in dem die einzelnen Schichten, also jeweils die Lineare Transformation mit Inputgröße und Outpuntdimension, sowie die nicht-lineare Aktivierungsfunktion
        prev = input_dim # Hier wird die Inputdimension initialisiert. Wird dann immer auf die Zieldimension der linearen Transformation gesetzt
        for h in hidden_dims: # Durchlauf durch alle Zieldimensionen der linearen Transformationen 
            layers += [nn.Linear(prev, h), nn.SiLU()] # Bildung der Schicht mit linearer Transformation(Inputgröße, Outputgröße) und Wahl der Aktivierungsfunktion
            prev = h # Outputdimension der vorherigen Sicht wird als Input für die nächste gesetzt
        self.net = nn.Sequential(*layers) # Sequential baut nun die Schichten als sequenzielle Pipeline zusammen, damit man beim Forwardpass einfach nurnoch den Input mit self.net(x) aufrufen muss und es durch Netz propagiert wird anstatt selbst den Input durch die einzelnen Schichten jagen zu müssen
        self.mu = nn.Linear(prev, latent_dim) # Nach der Featureextraktion durch das Netz wird nun eine eigene lineare Transformation genutzt um den Mittelwert vorherzusagen. Hier ist die Zieloutputdimension die festgelegte Dimension des Latenten Raums.
        self.logvar = nn.Linear(prev, latent_dim) # Gleiches geht für die Varianz, da Varianzen jedoch immer positiv sein müssen, aber eine lineare Transformation auch negative Zahlen ausspucken kann wird immer die logvar verwendet. Das Netz lernt das dann implizit weil im Reparametrisierungstrick die logvar in eine Standardabweichung umgewandelt wird zum Samples ziehen.

    def forward(self, x):
        h = self.net(x) # Hier wird dann einfach nur noch der Input durch das Netz geschoben, weil net ja die ganze Netsequenz enthält und der h-Vektor enthält dann die extrahierten Features, damit man sie in einer finalen auf das Target ausgerichteten Transformation verwenden kann
        return self.mu(h), self.logvar(h) # Nutzt die vorher initierte lineare Transformation um einen Mittelwertvektor und Logvar Vektor für das Sampling zu predicten. 


class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dims, output_dim):
        super().__init__() #Wieder der Konstruktoraufruf für das nn.Module zur Vererbung und initialisierung wichtiger Parts
        layers = [] # Gleich wie Encoder
        prev = latent_dim # Bekommt ein Sample aus dem Latenten Raum als Input 
        for h in hidden_dims: # Hidden Dims ist genau die umgedrehte Reihenfolge wie vom Encoder
            layers += [nn.Linear(prev, h), nn.SiLU()] #Gleich wie Encoder
            prev = h # Gleich wie Encoder
        layers += [nn.Linear(prev, output_dim)] # Zur finalen Rekonstruktion der Firma wird auch einfach eine lineare Transformation mit Zieldimension Anzahl der Features verwendet um den Input zu rekonstruieren
        self.net = nn.Sequential(*layers) # Decoder wird in einem Sequential Objekt als Netz gespeichert

    def forward(self, z):
        return self.net(z) # Schickt das Latente Embedding einmal durch alle Decoder Schichten und Spuckt die Rekonstruktion dann wieder aus. Hier wird direkt der Skalierte Input und nicht die Embeddings rekonstruiert. 

# Hier wird nun der Gesamte VAE mit den Feature Embeddings, dem Encoder, Decoder und der Reparametrisierung zusammengebaut.
class MissingValueVAE(nn.Module):
    def __init__(self, num_features, emb_dim=16, hidden_dims=(256, 128, 64), latent_dim=32): # Übergabe der Parameter Anzahl der Features, Dimension des Input Embeddings, die Dimensionen der Encoder/Decoder und die Dimension des Latenten Raums
        super().__init__() # Initialisierung der Mutterklasse nn.Module
        self.feature_embedding = FeatureEmbedding(num_features, emb_dim) #Erstellt ein Objekt, was die Linearen Transformationen zur Erstellung der Feature Embeddings enthält
        input_dim = num_features * emb_dim # Berechnung der Inputdimension für den Encoder, da die Input Embeddings geflattet und aufgereiht als Input übergeben werden.
        self.encoder = Encoder(input_dim, list(hidden_dims), latent_dim) #Initialisiert den Encoder
        self.decoder = Decoder(latent_dim, list(reversed(hidden_dims)), num_features) # Initialisiert den Decoder
# Die Reparametrisierung wird benötigt, weil man eine Zufallsprozess wie das samplen nicht ableiten kann, weil da ja keine Variablen durchgereicht werden können. Deshalb wird das Sampling als mü + eps * Std dargestellt, somit hat man eine ableitbare Form und der Zufallsprozess steckt in einer Konstanten eps, damit es differenzierbar bleibt.
    def reparameterize(self, mu, logvar): # Ohne diese Reparametrisierung und das damit einhergehende Sampling wäre der VAE nur ein normaler Autoencoder. Durch das Rauschen wird der Embeddingraum gefüllt und ermöglicht es so auch neue Daten zu generieren
        std = torch.exp(0.5 * logvar) # Umwandlung der log Varianz in eine Standardabweichung, damit mit Hilfe dieser ein Input für den Decoder berechnet werden kann. 
        eps = torch.randn_like(std) # Ziehung eines zufälligen eps damit ein Vektor 
        return mu + eps * std # Hier wird dann der tatsächliche Input für den Decoder berechnet. 

    def forward(self, x, mask):
        emb = self.feature_embedding(x, mask) #Hier wird dann die Feature Embedding Klasse verwendet um aus den Inputdaten mit Hilfe der Maske Input Embeddings zu machen. Fehlende Werte werden einfach durch das Missing Embedding ersetzt
        mu, logvar = self.encoder(emb) # Die Input Embeddings werden durch den Encoder geschoben, welcher einen Mittelwertvektor und einen Logvar Vektor zurückgeben
        z = self.reparameterize(mu, logvar) #Durch die Reparametrisierung wird mit Hilfe von Mü und Std ein Sample gezogen, was dann der Input für den Decoder ist.
        x_hat = self.decoder(z) # z wird durch den Decoder geschoben und gibt einen Rekonstruierten Input zurück 
        return x_hat, mu, logvar # Die Rekonstruktion, der Mittelwert, sowie die Logvar werden zurückgegeben um den Loss zu berechnen.


def masked_vae_loss(x, x_hat, mask, mu, logvar, beta=2.0, free_bits=0.0):
    # Reconstruction: nur über beobachtete Zellen (mask == 1)
    sq_error = (x - x_hat) ** 2 * mask # Hier wird der Squared Error berechnet also die quadrierte Differenz zwischen Rekonstruktion und tatsächlichem Input für jedes Element im Vektor. Anschließend wird die Maske multiplziert um den Loss Werte die durch ein Missing Embedding aufgefüllt wurden auf 0 zu setzen.
    recon_loss = sq_error.sum() / mask.sum().clamp(min=1.0) # Hier wird dann aus dem Squared Error der MSE weil die Summe der einzelnen Fehler durch die Anzahl der tatsächlichen Beaobachtungen geteilt wird. Das clamp verhinder eine Division durch 0.

    #  Hier wird für jede Dimension des Latenten Raums und jedes Beispiel die KL-Div zur Standardnormalverteilung berechnet. (batch, latent_dim)
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())  # logvar verhindert, dass eine zu kleine oder negative Varianz entsteht, da sie für kleine Werte gegen -inf geht. -mü^2 ist am kleinsten wenn mü=0, -exp(logvar) sorgt dafür das zu große Varianzen massiv bestraft werden. +1 ist ein Ausgleich, damit Loss genau 0 wenn mü=0 und std=1  
    if free_bits > 0:
        # "Free bits": jede Dimension darf mindestens so viel KL "kostenlos" haben,
        # bevor der Gradient sie weiter Richtung 0 drückt -> verhindert Posterior Collapse
        kl_per_dim = torch.clamp(kl_per_dim, min=free_bits)
    kl = kl_per_dim.sum(dim=1).mean()

    return recon_loss + beta * kl, recon_loss.item(), kl.item() # Hier werden als erstes der Tensor mit dem gesamten Rechengraphen übergeben, damit der Gradient berechnet werden kann. Mit der Item() funktion wird ein einzelnes Skalar als Loss zum Logging übergeben. Da der Rechengraph wo an den Tensoren hängt sehr speicherintensiv ist.


def train_vae(
    X_train, M_train, X_val, M_val,
    emb_dim=16, hidden_dims=(256, 128, 64), latent_dim=32,
    batch_size=256, lr=1e-3, epochs=100, beta=1.0, device=None,
    print_every=5, eval_every=10, feature_names=None,
    kl_warmup_epochs=0, free_bits=0.0,
):
    """
    eval_every: alle wie viele Epochen zusätzlich der Imputations-Check
        (evaluate_imputation_quality) auf den Val-Daten mitläuft und ausgegeben
        wird. 0/None deaktiviert das (spart etwas Zeit bei sehr großen Daten).
    kl_warmup_epochs: linear von beta=0 auf beta=<beta> über die ersten N Epochen
        hochfahren, statt von Anfang an den vollen KL-Druck anzuwenden. Hilft
        gegen Posterior Collapse (Encoder ignoriert den Input, KL fällt sofort
        auf ~0). 0 = kein Warmup.
    free_bits: Mindest-KL pro Latent-Dimension (z.B. 0.5), unterhalb dessen kein
        Gradient mehr Richtung "noch kleiner" wirkt. Zusätzliche Absicherung
        gegen Posterior Collapse, kombinierbar mit kl_warmup_epochs.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    train_ds = PanelDataset(X_train, M_train)
    val_ds = PanelDataset(X_val, M_val)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = MissingValueVAE(
        num_features=X_train.shape[1], emb_dim=4,
        hidden_dims=hidden_dims, latent_dim=latent_dim,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [], "train_recon": [], "train_kl": [],
        "val_loss": [], "val_recon": [], "val_kl": [],
        "val_imputation_mse": [],  # Liste von (epoch, mse)
        "beta": [],
    }

    for epoch in range(1, epochs + 1):
        current_beta = beta * min(1.0, epoch / kl_warmup_epochs) if kl_warmup_epochs > 0 else beta
        history["beta"].append(current_beta)

        model.train()
        train_loss = train_recon = train_kl = 0.0
        for x, m in train_loader:
            x, m = x.to(device), m.to(device)
            optimizer.zero_grad()
            x_hat, mu, logvar = model(x, m)
            loss, recon, kl = masked_vae_loss(x, x_hat, m, mu, logvar, beta=current_beta, free_bits=free_bits)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
            train_recon += recon * x.size(0)
            train_kl += kl * x.size(0)
        n_train = len(train_ds)
        train_loss /= n_train
        train_recon /= n_train
        train_kl /= n_train

        model.eval()
        val_loss = val_recon = val_kl = 0.0
        with torch.no_grad():
            for x, m in val_loader:
                x, m = x.to(device), m.to(device)
                x_hat, mu, logvar = model(x, m)
                loss, recon, kl = masked_vae_loss(x, x_hat, m, mu, logvar, beta=current_beta, free_bits=free_bits)
                val_loss += loss.item() * x.size(0)
                val_recon += recon * x.size(0)
                val_kl += kl * x.size(0)
        n_val = len(val_ds)
        val_loss /= n_val
        val_recon /= n_val
        val_kl /= n_val

        history["train_loss"].append(train_loss)
        history["train_recon"].append(train_recon)
        history["train_kl"].append(train_kl)
        history["val_loss"].append(val_loss)
        history["val_recon"].append(val_recon)
        history["val_kl"].append(val_kl)

        if print_every and (epoch % print_every == 0 or epoch == 1):
            print(
                f"Epoch {epoch:3d} | "
                f"train: loss={train_loss:.4f} recon={train_recon:.4f} kl={train_kl:.4f} | "
                f"val: loss={val_loss:.4f} recon={val_recon:.4f} kl={val_kl:.4f}"
            )

        if eval_every and epoch % eval_every == 0:
            imp_mse, _, _ = evaluate_imputation_quality(model, X_val, M_val, device=device)
            history["val_imputation_mse"].append((epoch, imp_mse))
            print(f"           -> Imputation-Check (val, künstlich versteckt): MSE={imp_mse:.4f}")

    print_final_summary(model, X_val, M_val, feature_names=feature_names, device=device)

    return model, history


def print_final_summary(model, X_val, M_val, feature_names=None, device=None, top_n=10):
    """Druckt eine Zusammenfassung aller Kernmetriken nach Trainingsende."""
    res = evaluate_reconstruction(model, X_val, M_val, device=device, feature_names=feature_names)
    print("\n=== Finale Evaluationsmetriken (Val-Set) ===")
    print(f"Overall Reconstruction-MSE (beobachtete Zellen): {res['overall_mse']:.4f}")

    if feature_names is not None:
        sorted_items = sorted(res["per_feature_mse"].items(), key=lambda kv: kv[1], reverse=True)
        print(f"Schlechteste {top_n} Features (höchster MSE):")
        for name, mse in sorted_items[:top_n]:
            print(f"  {name}: {mse:.4f}")

    imp_mse, true_vals, pred_vals = evaluate_imputation_quality(model, X_val, M_val, device=device)
    print(f"Imputation-Check MSE (künstlich versteckte, aber bekannte Werte): {imp_mse:.4f}")
    print("=============================================\n")


@torch.no_grad()
def evaluate_reconstruction(model, X, M, device=None, feature_names=None):
    """
    Maskierter Reconstruction-Error (MSE) auf tatsächlich BEOBACHTETEN Werten.
    Das ist die einzige Stelle, an der man direkt gegen Ground Truth prüfen kann -
    für echte Missing-Werte gibt es ja keinen wahren Wert zum Vergleichen.
    Gibt overall MSE + MSE pro Feature zurück (zeigt, welche Items gut/schlecht
    modelliert werden).
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    x = torch.as_tensor(X, dtype=torch.float32).to(device)
    m = torch.as_tensor(M, dtype=torch.float32).to(device)

    x_hat, mu, logvar = model(x, m)
    sq_error = (x - x_hat) ** 2 * m

    overall_mse = (sq_error.sum() / m.sum().clamp(min=1.0)).item()
    per_feature_mse = (sq_error.sum(dim=0) / m.sum(dim=0).clamp(min=1.0)).cpu().numpy()

    result = {"overall_mse": overall_mse, "per_feature_mse": per_feature_mse}
    if feature_names is not None:
        result["per_feature_mse"] = dict(zip(feature_names, per_feature_mse))
    return result


@torch.no_grad()
def get_latent_embeddings(model, X, M, device=None):
    """
    Deterministische Embeddings (mu, nicht das gesampelte z) pro Zeile -
    das ist das, was du später z.B. als Node-Features für den GAT nehmen würdest.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    x = torch.as_tensor(X, dtype=torch.float32).to(device)
    m = torch.as_tensor(M, dtype=torch.float32).to(device)

    emb = model.feature_embedding(x, m)
    mu, _ = model.encoder(emb)
    return mu.cpu().numpy()


def evaluate_imputation_quality(model, X, M, device=None, mask_fraction=0.1, seed=0):
    """
    Versteckt zusätzlich einen Teil der ohnehin beobachteten Werte künstlich und
    prüft, wie gut das Modell genau diese (bekannten!) Werte rekonstruiert.
    Das ist der einzige Weg, echte Imputationsqualität mit Ground Truth zu messen -
    denn bei "natürlich" fehlenden Werten weißt du nie, was der wahre Wert war.
    """
    rng = np.random.default_rng(seed)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    observed_idx = np.argwhere(M == 1)
    n_to_hide = int(len(observed_idx) * mask_fraction)
    hide_idx = observed_idx[rng.choice(len(observed_idx), n_to_hide, replace=False)]

    M_eval = M.copy()
    M_eval[hide_idx[:, 0], hide_idx[:, 1]] = 0  # zusätzlich verstecken (Modell sieht diese nicht)

    model.eval().to(device)
    with torch.no_grad():
        x = torch.as_tensor(X, dtype=torch.float32).to(device)
        m_eval = torch.as_tensor(M_eval, dtype=torch.float32).to(device)
        x_hat, _, _ = model(x, m_eval)

    true_vals = X[hide_idx[:, 0], hide_idx[:, 1]]
    pred_vals = x_hat.cpu().numpy()[hide_idx[:, 0], hide_idx[:, 1]]
    mse = float(np.mean((true_vals - pred_vals) ** 2))
    return mse, true_vals, pred_vals


if __name__ == "__main__":
    # X_train, M_train, X_val, M_val kommen aus deiner Datenvorbereitung:
    # X = scale(df).fillna(0.0).values
    # M = df[usable_features].notna().astype(int).values
    model, history = train_vae(
        X_train_scaled, X_train_mask, X_val_scaled, X_val_mask,
        epochs=200, latent_dim=32, feature_names=usable_features,
        kl_warmup_epochs=25, free_bits=0.2, beta=2.0,
    )
    torch.save(model.state_dict(), "missing_value_vae.pt")

Epoch   1 | train: loss=2.1355 recon=1.6124 kl=6.5379 | val: loss=1.8979 recon=1.3647 kl=6.6650
Epoch   5 | train: loss=3.6936 recon=1.1253 kl=6.4206 | val: loss=3.6524 recon=1.0836 kl=6.4220
Epoch  10 | train: loss=6.1085 recon=0.9819 kl=6.4082 | val: loss=6.0842 recon=0.9508 kl=6.4168
           -> Imputation-Check (val, künstlich versteckt): MSE=1.3908
Epoch  15 | train: loss=8.6101 recon=0.9251 kl=6.4042 | val: loss=8.5883 recon=0.9029 kl=6.4045
Epoch  20 | train: loss=11.1387 recon=0.8943 kl=6.4028 | val: loss=11.1262 recon=0.8822 kl=6.4025
           -> Imputation-Check (val, künstlich versteckt): MSE=1.3616
Epoch  25 | train: loss=13.6844 recon=0.8808 kl=6.4018 | val: loss=13.6730 recon=0.8704 kl=6.4013
Epoch  30 | train: loss=13.6751 recon=0.8713 kl=6.4019 | val: loss=13.6576 recon=0.8554 kl=6.4011
           -> Imputation-Check (val, künstlich versteckt): MSE=1.3495
Epoch  35 | train: loss=13.6634 recon=0.8595 kl=6.4019 | val: loss=13.6403 recon=0.8361 kl=6.4021
Epoch  40 | tr

In [7]:
import inspect
print(inspect.signature(train_vae))

(X_train, M_train, X_val, M_val, emb_dim=16, hidden_dims=(256, 128, 64), latent_dim=32, batch_size=256, lr=0.001, epochs=100, beta=1.0, device=None, print_every=5, eval_every=10, feature_names=None, kl_warmup_epochs=0, free_bits=0.0)


In [6]:
pd.DataFrame(X_train_scaled).std().mean()

0.6306462